# Global Superstore Questions Assignment 2.3 and 2.4
Scott Person

In [0]:
data_path = './data/'

In [0]:
%pip install openpyxl

import numpy as np
import pandas as pd

## Load Data
Load both Excel sheets into pandas dataframes

In [0]:
orders_df = pd.read_excel(data_path + 'Global_Superstore_Orders_2012_2015.xlsx', sheet_name='Orders')
people_df = pd.read_excel(data_path + 'Global_Superstore_Orders_2012_2015.xlsx', sheet_name='People')

## Print Shape
Print the shape of both dataframes including memory usage for Orders

### Orders

In [0]:
print(f'Shape: {orders_df.shape}')

print(f'Size: {orders_df.memory_usage(deep=True).sum():,} bytes')

### People


In [0]:
print(f'Shape: {people_df.shape}')

## Apply Snake

In [0]:
def to_snake(columns):
    'Lowercase, turn anything that is not a letter or digit into _, trim stray underscores.'
    return (pd.Index(columns)
            .str.strip()
            .str.lower()
            .str.replace(r"[^0-9a-z]+", "_", regex=True)
            .str.strip("_"))

In [0]:
people_df.columns = to_snake(people_df.columns)

orders_columns_orig = orders_df.columns
orders_df.columns = to_snake(orders_columns_orig)

pd.DataFrame({"before": orders_columns_orig, "after": orders_df.columns})

In [0]:
def dataframe_schema_markdown(df):
    """Return a markdown table listing each column's name, current dtype,
    and blank columns for a proposed new datatype, and the reason."""
    rows = [
        f"| {i} | {col} | {str(dtype)} |  |  |"
        for i, (col, dtype) in enumerate(df.dtypes.items(), start=1)
    ]
    header = "| # | column_name | datatype | new_datatype | reason |\n|---|---|---|---|---|\n"
    return header + "\n".join(rows)

print(dataframe_schema_markdown(orders_df))

In [0]:
orders_df.nunique()

In [0]:
orders_df.head()

| # | column_name | datatype | new_datatype | reason |
|---|---|---|---|---|
| 1 | row_id | int64 |int64 |this is a counter; do not use for analysis  |
| 2 | order_id | object |object  |natural key of the table  |
| 3 | order_date | datetime64[ns] |datetime64  |this is a date  |
| 4 | ship_date | datetime64[ns] |datetime64  |this is a date  |
| 5 | ship_mode | object |category  |small, fixed set of values  |
| 6 | customer_id | object |object  |this is an identifier - likely a foreign key  |
| 7 | customer_name | object |object  |high cardinality text; high cardinality data is not conducive to categories; not a common analytical subject  |
| 8 | segment | object |category  |small fixed set of values  |
| 9 | postal_code | float64 |object  |postal codes often have leading zeros which a numerical type will remove; we don't do math on postal codes  |
| 10 | city | object |object  |high cardinality text  |
| 11 | state | object |category |(relatively) fixed set of values with low cardinality  |
| 12 | country | object |category  |as with state, relatively fixed with low cardinality  |
| 13 | region | object |category |low cardinality text; we will likely sort and group on this; this holds for the other category columns as well  |
| 14 | market | object |category  |low cardinality text  |
| 15 | product_id | object |object  |this is an identifier - likely a foreign key  |
| 16 | category | object |category  |low cardinality text  |
| 17 | sub_category | object |category  |low cardinality text  |
| 18 | product_name | object |object  |relatively low cardinality but subject to change  |
| 19 | sales | float64 |int64  |going old school and storing all of my dollar values as int but converting at output to two decimals; this will eliminate floating point math issues; a scale value will be stored in a constant to allow conversion from int back to dollars and cents  |
| 20 | quantity | int64 |int64  |it's a number of things so int makes sense  |
| 21 | discount | float64 |float64  |this is a decimal representation of a percentage  |
| 22 | profit | float64 |int64  |sticking with int for my amounts  |
| 23 | shipping_cost | float64 |int64  |sticking with int for my amounts  |
| 24 | order_priority | object |category - ordered  |small fixed set of values; has a natrual order  |


In [0]:
MONEY_SCALE = 10000

# --- Convert columns to the datatypes specified in the schema table ---

print(f"Memory usage before: {orders_df.memory_usage(deep=True).sum():,} bytes")

# Category columns (unordered)
for col in ['ship_mode', 'segment', 'state', 'country', 'region',
           'market', 'category', 'sub_category']:
    orders_df[col] = orders_df[col].astype('category')

# Ordered category for order_priority (Low < Medium < High < Critical)
orders_df['order_priority'] = pd.Categorical(
    orders_df['order_priority'],
    categories=['Low', 'Medium', 'High', 'Critical'],
    ordered=True
)

# Postal code: float64 -> object (string), stripping the trailing '.0'
# Padding to 5 digits with leading zeros to preserve values like 02101
orders_df['postal_code'] = (
    orders_df['postal_code']
        .dropna()
        .astype('Int64')      # nullable int so we can go to string cleanly
        .astype('string')
        .str.zfill(5)          # pad with leading zeros to 5 digits
        .reindex(orders_df.index)
)

# Money columns: float64 -> int64 (scaled to eliminate floating-point issues)
for col in ['sales', 'profit', 'shipping_cost']:
    orders_df[col] = (orders_df[col] * MONEY_SCALE).round().astype('int64')

print(f"Memory usage after:  {orders_df.memory_usage(deep=True).sum():,} bytes")
orders_df.dtypes



In [0]:
print(f"Before merge — orders: {len(orders_df):,}  people: {len(people_df):,}")

orders_merged_df = orders_df.merge(people_df, on='region', how='left', indicator=True, validate='many_to_one')

print(f"After merge  — merged:  {len(orders_merged_df):,}")

print(orders_merged_df.groupby(['region', '_merge']).size())

left_only = orders_merged_df.loc[orders_merged_df['_merge'] == 'left_only']

print(f"\nTotal sales (left only): {left_only['sales'].sum() / MONEY_SCALE:,.2f}")